<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/Implementation_of_QuantV2X.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [CELL 1 - Dependencies & System Setup]
import os
import io
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Set seed for exact reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"QuantV2X Execution Engine Initialized on Device: {device}")

QuantV2X Execution Engine Initialized on Device: cuda


In [2]:
# [CELL 2 - Multi-Agent BEV Dataset with Occlusion Modeling]

class SyntheticV2XBEVDataset(Dataset):
    """
    Simulates Bird's-Eye-View (BEV) perception for Ego Vehicle (Agent 1)
    and Connected Vehicle / Infrastructure (Agent 2).
    Models occlusions where Ego misses objects that Agent 2 clearly observes.
    """
    def __init__(self, num_samples=500, grid_size=32, num_channels=16):
        self.num_samples = num_samples
        self.grid_size = grid_size
        self.num_channels = num_channels
        self.data = [self._generate_sample() for _ in range(num_samples)]

    def _generate_sample(self):
        H = W = self.grid_size

        # Ground-truth object occupancy mask (3D BEV bounding target)
        gt_occupancy = np.zeros((1, H, W), dtype=np.float32)

        # Agent 1 (Ego) raw BEV feature map
        feat_ego = np.random.normal(0, 0.1, (self.num_channels, H, W)).astype(np.float32)

        # Agent 2 (Connected Agent) raw BEV feature map
        feat_agent2 = np.random.normal(0, 0.1, (self.num_channels, H, W)).astype(np.float32)

        # Inject 2-4 objects into the BEV scene
        num_objects = random.randint(2, 4)
        for _ in range(num_objects):
            r = random.randint(4, H - 5)
            c = random.randint(4, W - 5)
            size = random.randint(2, 4)

            gt_occupancy[0, r:r+size, c:c+size] = 1.0

            # Object feature pattern
            pattern = np.random.uniform(0.8, 1.5, (self.num_channels, size, size))

            # Agent 2 sees all objects clearly
            feat_agent2[:, r:r+size, c:c+size] += pattern

            # Agent 1 experiences occlusion on 50% of objects
            if random.random() > 0.5:
                feat_ego[:, r:r+size, c:c+size] += pattern # Visible to Ego
            # Otherwise occluded for Ego (feat_ego stays low)

        return (
            torch.tensor(feat_ego),
            torch.tensor(feat_agent2),
            torch.tensor(gt_occupancy)
        )

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = SyntheticV2XBEVDataset(num_samples=600)
test_dataset  = SyntheticV2XBEVDataset(num_samples=150)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Synthetic V2X Dataset Ready: Train={len(train_dataset)} | Test={len(test_dataset)}")

Synthetic V2X Dataset Ready: Train=600 | Test=150


In [3]:
# [CELL 3 - Straight-Through Estimator (STE) Quantization Engine]

class QuantizeSTE(torch.autograd.Function):
    """
    Simulates uniform symmetric N-bit quantization during forward pass
    while preserving continuous gradient flow via Straight-Through Estimator (STE).
    """
    @staticmethod
    def forward(ctx, x, num_bits=8):
        qmin = -(2 ** (num_bits - 1))
        qmax = (2 ** (num_bits - 1)) - 1

        # Calculate per-tensor dynamic scale
        max_val = torch.max(torch.abs(x))
        scale = max_val / float(qmax) + 1e-8

        # Quantize and clamp
        x_q = torch.clamp(torch.round(x / scale), qmin, qmax)

        # Dequantize back to float
        x_deq = x_q * scale
        return x_deq

    @staticmethod
    def backward(ctx, grad_output):
        # STE passes gradient back directly
        return grad_output, None

def quantize_features(x, num_bits=8):
    if num_bits >= 32:
        return x
    return QuantizeSTE.apply(x, num_bits)

def compute_payload_bytes(tensor, num_bits=8):
    """Calculates exact network transmission size in Bytes."""
    num_elements = tensor.numel()
    bytes_per_elem = num_bits / 8.0
    # Include tiny 8-byte scale metadata overhead
    return int(num_elements * bytes_per_elem) + 8

print("Quantization Engine Loaded (Supports FP32, INT8, INT4).")

Quantization Engine Loaded (Supports FP32, INT8, INT4).


In [4]:
# [CELL 4 - QuantV2X Multi-Agent Fusion Architecture]

class CrossAgentAttentionalFusion(nn.Module):
    """
    Fuses Ego BEV features with dequantized Connected Agent feature messages
    using spatial cross-attention.
    """
    def __init__(self, in_channels=16):
        super().__init__()
        self.query_conv = nn.Conv2d(in_channels, in_channels // 2, kernel_size=1)
        self.key_conv   = nn.Conv2d(in_channels, in_channels // 2, kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.gamma      = nn.Parameter(torch.zeros(1))

    def forward(self, ego_feat, agent_feat):
        B, C, H, W = ego_feat.shape
        proj_query = self.query_conv(ego_feat).view(B, -1, H * W).permute(0, 2, 1) # [B, N, C']
        proj_key   = self.key_conv(agent_feat).view(B, -1, H * W)                  # [B, C', N]

        energy = torch.bmm(proj_query, proj_key)                                    # [B, N, N]
        attention = F.softmax(energy, dim=-1)

        proj_value = self.value_conv(agent_feat).view(B, -1, H * W)                 # [B, C, N]
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(B, C, H, W)

        return ego_feat + self.gamma * out

class QuantV2XReasoner(nn.Module):
    """
    Complete QuantV2X Multi-Agent Intermediate Fusion Perception Pipeline.
    """
    def __init__(self, in_channels=16, quant_bits=8):
        super().__init__()
        self.quant_bits = quant_bits

        # Local feature refinement encoders
        self.ego_encoder   = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, padding=1)
        )
        self.agent_encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, padding=1)
        )

        self.fusion = CrossAgentAttentionalFusion(in_channels=16)

        # BEV Object Detection Head
        self.detection_head = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, feat_ego, feat_agent2, override_bits=None):
        bits = override_bits if override_bits is not None else self.quant_bits

        f_ego = self.ego_encoder(feat_ego)
        f_agent = self.agent_encoder(feat_agent2)

        # Simulate Message Transmit over V2X Link (Quantize transmitted message)
        f_agent_transmitted = quantize_features(f_agent, num_bits=bits)

        # Ego fuses local features with received compressed message
        fused_feat = self.fusion(f_ego, f_agent_transmitted)

        # Predict BEV Object Occupancy Grid
        bev_pred = self.detection_head(fused_feat)
        return bev_pred

model = QuantV2XReasoner(quant_bits=8).to(device)
print(f"QuantV2X Multi-Agent Model Loaded Successfully.")

QuantV2X Multi-Agent Model Loaded Successfully.


In [5]:
# [CELL 5 - Quantization-Aware Training Loop]

def compute_bev_iou(pred_mask, gt_mask, threshold=0.5):
    """Computes Mean Intersection over Union (mIoU) for BEV Object Detection."""
    pred_bin = (pred_mask >= threshold).float()
    intersection = (pred_bin * gt_mask).sum(dim=(1, 2, 3))
    union = torch.clamp((pred_bin + gt_mask).sum(dim=(1, 2, 3)) - intersection, min=1e-6)
    return (intersection / union).mean().item()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
bce_loss = nn.BCELoss()

epochs = 20
print("Starting QuantV2X Quantization-Aware Training (QAT)...")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_iou = 0.0, 0.0

    for batch_ego, batch_agent2, batch_gt in train_loader:
        batch_ego = batch_ego.to(device)
        batch_agent2 = batch_agent2.to(device)
        batch_gt = batch_gt.to(device)

        optimizer.zero_grad()

        # Train with INT8 Quantized Intermediate Messages
        bev_pred = model(batch_ego, batch_agent2, override_bits=8)
        loss = bce_loss(bev_pred, batch_gt)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_iou += compute_bev_iou(bev_pred, batch_gt)

    if epoch % 4 == 0 or epoch == 1:
        avg_loss = total_loss / len(train_loader)
        avg_iou = (total_iou / len(train_loader)) * 100.0
        print(f"Epoch [{epoch:02d}/{epochs}] | BCE Loss: {avg_loss:.4f} | BEV Detection mIoU: {avg_iou:.2f}%")

Starting QuantV2X Quantization-Aware Training (QAT)...
Epoch [01/20] | BCE Loss: 0.4395 | BEV Detection mIoU: 30.79%
Epoch [04/20] | BCE Loss: 0.1493 | BEV Detection mIoU: 49.24%
Epoch [08/20] | BCE Loss: 0.0828 | BEV Detection mIoU: 49.35%
Epoch [12/20] | BCE Loss: 0.0705 | BEV Detection mIoU: 49.24%
Epoch [16/20] | BCE Loss: 0.0650 | BEV Detection mIoU: 49.08%
Epoch [20/20] | BCE Loss: 0.0609 | BEV Detection mIoU: 48.69%


In [6]:
# [CELL 6 - Comprehensive System Benchmark]

def evaluate_system():
    model.eval()

    metrics = {
        "Single-Agent (No V2X)": {"iou": [], "payload_bytes": 0},
        "Full Precision (FP32 V2X)": {"iou": [], "payload_bytes": 0},
        "QuantV2X (INT8 Ours)": {"iou": [], "payload_bytes": 0},
        "QuantV2X (INT4 Ours)": {"iou": [], "payload_bytes": 0},
    }

    sample_feat_shape = None

    with torch.no_grad():
        for batch_ego, batch_agent2, batch_gt in test_loader:
            batch_ego = batch_ego.to(device)
            batch_agent2 = batch_agent2.to(device)
            batch_gt = batch_gt.to(device)

            # 1. Single Agent (Zeros transmitted from Agent 2)
            pred_single = model(batch_ego, torch.zeros_like(batch_agent2), override_bits=32)
            metrics["Single-Agent (No V2X)"]["iou"].append(compute_bev_iou(pred_single, batch_gt))

            # 2. FP32 Full Precision
            pred_fp32 = model(batch_ego, batch_agent2, override_bits=32)
            metrics["Full Precision (FP32 V2X)"]["iou"].append(compute_bev_iou(pred_fp32, batch_gt))

            # 3. QuantV2X INT8
            pred_int8 = model(batch_ego, batch_agent2, override_bits=8)
            metrics["QuantV2X (INT8 Ours)"]["iou"].append(compute_bev_iou(pred_int8, batch_gt))

            # 4. QuantV2X INT4
            pred_int4 = model(batch_ego, batch_agent2, override_bits=4)
            metrics["QuantV2X (INT4 Ours)"]["iou"].append(compute_bev_iou(pred_int4, batch_gt))

    # Single transmission payload size calculation
    dummy_msg = torch.randn(1, 16, 32, 32)
    metrics["Single-Agent (No V2X)"]["payload_bytes"] = 0
    metrics["Full Precision (FP32 V2X)"]["payload_bytes"] = compute_payload_bytes(dummy_msg, 32)
    metrics["QuantV2X (INT8 Ours)"]["payload_bytes"] = compute_payload_bytes(dummy_msg, 8)
    metrics["QuantV2X (INT4 Ours)"]["payload_bytes"] = compute_payload_bytes(dummy_msg, 4)

    fp32_bytes = metrics["Full Precision (FP32 V2X)"]["payload_bytes"]
    fp32_iou = np.mean(metrics["Full Precision (FP32 V2X)"]["iou"]) * 100.0
    int8_iou = np.mean(metrics["QuantV2X (INT8 Ours)"]["iou"]) * 100.0
    single_iou = np.mean(metrics["Single-Agent (No V2X)"]["iou"]) * 100.0

    print("\n" + "=" * 68)
    print("QUANTV2X SYSTEM BENCHMARK EVALUATION RESULTS")
    print("=" * 68)
    print(f"{'Perception System':<28} | {'BEV mIoU':<10} | {'Payload Size':<12} | {'Bandwidth Saved':<12}")
    print("-" * 68)
    for name, data in metrics.items():
        mean_iou = np.mean(data["iou"]) * 100.0
        p_bytes = data["payload_bytes"]
        p_str = f"{p_bytes / 1024:.2f} KB" if p_bytes > 0 else "0 KB"
        saved = f"{(1.0 - p_bytes / fp32_bytes)*100:.1f}%" if fp32_bytes > 0 else "N/A"
        print(f"{name:<28} | {mean_iou:>8.2f}% | {p_str:>12} | {saved:>12}")
    print("=" * 68)

    net_improvement = int8_iou - single_iou
    accuracy_retention = (int8_iou / fp32_iou) * 100.0
    print(f"Net Improvement over Single-Agent Ego       : +{net_improvement:.2f} pp")
    print(f"FP32 Accuracy Retention under INT8 Constraint : {accuracy_retention:.1f}%")
    print(f"Transmission Payload Bandwidth Reduction    : 75.0% (4x Compression)")

evaluate_system()


QUANTV2X SYSTEM BENCHMARK EVALUATION RESULTS
Perception System            | BEV mIoU   | Payload Size | Bandwidth Saved
--------------------------------------------------------------------
Single-Agent (No V2X)        |    51.21% |         0 KB |       100.0%
Full Precision (FP32 V2X)    |    51.60% |     64.01 KB |         0.0%
QuantV2X (INT8 Ours)         |    51.60% |     16.01 KB |        75.0%
QuantV2X (INT4 Ours)         |    51.60% |      8.01 KB |        87.5%
Net Improvement over Single-Agent Ego       : +0.39 pp
FP32 Accuracy Retention under INT8 Constraint : 100.0%
Transmission Payload Bandwidth Reduction    : 75.0% (4x Compression)
